[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_14/simple_eda_california_housing.ipynb)

# Einfache explorative Datenanalyse: California Housing

In diesem Notebook untersuchen wir den California-Housing-Datensatz Schritt für Schritt. Eine Zeile beschreibt eine **Zensus-Blockgruppe** aus dem Zensus von 1990, nicht ein einzelnes Haus.

`target` ist der Median-Hauswert in Einheiten von **100.000 US-Dollar**. `MedInc` ist das mittlere Haushaltseinkommen in Einheiten von **10.000 US-Dollar**. Die gespeicherten Hauswerte sind bei `5.00001` nach oben gekappt. Deshalb kennen wir den tatsächlichen höchsten Median-Hauswert nicht. Beim ersten Laden des Datensatzes wird eine Internetverbindung benötigt.

Führe die Codezellen der Reihe nach aus. In Jupyter kannst du dafür eine Zelle auswählen und **Umschalt+Enter** drücken.

## 1. Bibliotheken laden

Pandas hilft uns beim Arbeiten mit Tabellen. Matplotlib erstellt Diagramme. Scikit-learn stellt den Datensatz bereit.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing

print("Bibliotheken geladen!")

## 2. Datensatz laden

Wir laden die Eingabemerkmale und ergänzen die Zielgröße `target`. Ein DataFrame ist Pandas' Tabellenformat.

In [ ]:
print("Lade California Housing Datensatz...")
housing = fetch_california_housing()

df = pd.DataFrame(housing.data, columns=housing.feature_names)
df["target"] = housing.target

print(f"Datensatz geladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")

## 3. Erste Informationen

Wir prüfen die Spalten, die Anzahl der Zeilen und Spalten sowie die Datentypen. So bekommen wir einen ersten Überblick über die Tabelle.

In [ ]:
print("Spalten im Datensatz:")
print(df.columns.tolist())

print(f"\nAnzahl Zeilen: {len(df)}")
print(f"Anzahl Spalten: {len(df.columns)}")

print("\nDatentypen:")
print(df.dtypes)

## 4. Deskriptive Statistiken

`describe()` fasst numerische Spalten zusammen. Dazu gehören zum Beispiel Mittelwert, Standardabweichung, Minimum und Maximum. Die Werte werden auf zwei Nachkommastellen gerundet.

In [ ]:
df.describe().round(2)

## 5. Fehlende Werte prüfen

Die Summe pro Spalte zeigt, wie viele Werte fehlen. Ein fehlender Wert wird in Pandas als `NaN` dargestellt.

In [ ]:
missing = df.isnull().sum()
print(missing)

if missing.sum() == 0:
    print("Keine fehlenden Werte gefunden!")

## 6. Daten visualisieren

Histogramme zeigen, wie häufig Wertebereiche vorkommen. Ein Streudiagramm zeigt den Zusammenhang zweier Merkmale: Jeder Punkt steht für eine Zensus-Blockgruppe.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Verteilung der gespeicherten Median-Hauswerte
axes[0, 0].hist(df["target"], bins=30, alpha=0.7, color="blue")
axes[0, 0].set_title("Verteilung der Immobilienpreise")
axes[0, 0].set_xlabel("Preis (100.000 USD)")
axes[0, 0].set_ylabel("Häufigkeit")

# Verteilung des mittleren Haushaltseinkommens
axes[0, 1].hist(df["MedInc"], bins=30, alpha=0.7, color="green")
axes[0, 1].set_title("Verteilung des mittleren Einkommens")
axes[0, 1].set_xlabel("Einkommen (10.000 USD)")
axes[0, 1].set_ylabel("Häufigkeit")

# Einkommen und Hauswert gegenüberstellen
axes[1, 0].scatter(df["MedInc"], df["target"], alpha=0.5, s=10)
axes[1, 0].set_title("Einkommen vs. Immobilienpreise")
axes[1, 0].set_xlabel("Mittleres Einkommen (10.000 USD)")
axes[1, 0].set_ylabel("Immobilienpreis (100.000 USD)")

# Verteilung der durchschnittlichen Zimmeranzahl
axes[1, 1].hist(df["AveRooms"], bins=30, alpha=0.7, color="red")
axes[1, 1].set_title("Durchschnittliche Zimmeranzahl")
axes[1, 1].set_xlabel("Anzahl Zimmer")
axes[1, 1].set_ylabel("Häufigkeit")

plt.tight_layout()
plt.show()

## 7. Extreme Hauswerte untersuchen

Wir suchen eine Zeile mit dem höchsten und eine mit dem niedrigsten gespeicherten Wert in `target`. Der Höchstwert ist im Datensatz gekappt. Er zeigt daher nicht, wie hoch der tatsächliche Median-Hauswert in dieser Region war.

In [ ]:
max_price_idx = df["target"].idxmax()
print(
    f"Eine Region mit höchstem gespeichertem Median-Hauswert: "
    f"${df.loc[max_price_idx, 'target'] * 100:.0f}k (gekappt)"
)
print(f"  Mittleres Einkommen: ${df.loc[max_price_idx, 'MedInc'] * 10:.0f}k")
print(f"  Durchschnittliche Zimmeranzahl: {df.loc[max_price_idx, 'AveRooms']:.1f}")

min_price_idx = df["target"].idxmin()
print(
    f"\nEine Region mit niedrigstem Median-Hauswert: "
    f"${df.loc[min_price_idx, 'target'] * 100:.0f}k"
)
print(f"  Mittleres Einkommen: ${df.loc[min_price_idx, 'MedInc'] * 10:.0f}k")
print(f"  Durchschnittliche Zimmeranzahl: {df.loc[min_price_idx, 'AveRooms']:.1f}")

## 8. Korrelationen mit dem Hauswert

Eine Korrelation beschreibt einen linearen Zusammenhang. Positive Werte bedeuten, dass beide Größen tendenziell gemeinsam steigen; negative Werte bedeuten, dass eine Größe tendenziell sinkt, wenn die andere steigt. Eine Korrelation beweist keine Ursache.

In [ ]:
correlations = df.corr()["target"].drop("target").sort_values(ascending=False)

print("Stärkste positive Korrelationen:")
for feature, corr in correlations[correlations > 0].head(3).items():
    print(f"  {feature}: {corr:.3f}")

print("\nStärkste negative Korrelationen:")
for feature, corr in correlations[correlations < 0].sort_values().head(3).items():
    print(f"  {feature}: {corr:.3f}")

## Übungen

Erweitere die Analyse, indem du für jede Aufgabe eine eigene Funktion schreibst und sie mit `df` ausprobierst.

### 1. Altersanalyse

Schreibe `analyze_house_age(df)`:
- Erstelle ein Histogramm des Merkmals `HouseAge`.
- Berechne den Mittelwert und den Höchstwert des regionalen Median-Gebäudealters. `HouseAge` ist ein Median pro Region und bei 52 gekappt.
- Berechne die Korrelation zwischen Gebäudealter und Hauswert.

Hilfreiche Ausdrücke: `plt.hist(df['HouseAge'], bins=30)`, `df['HouseAge'].mean()`, `df['HouseAge'].max()`, `df.corr()['target']['HouseAge']`.

### 2. Geografische Verteilung

Schreibe `geographic_analysis(df)`:
- Erstelle ein Streudiagramm von `Longitude` gegen `Latitude`.
- Färbe die Punkte nach Hauswert ein (teuer = rot, günstig = blau).
- Finde die nördlichste und südlichste Region.

Hilfreiche Ausdrücke: `plt.scatter(df['Longitude'], df['Latitude'], c=df['target'], cmap='RdBu_r')`, `plt.colorbar()`, `df['Latitude'].max()`, `df['Latitude'].min()`, `df.loc[df['Latitude'].idxmax()]`.

### 3. Bevölkerungszahl

Schreibe `population_analysis(df)`:
- Berechne die Korrelation zwischen `Population` und Hauswert.
- Erstelle ein Histogramm der Population.
- Finde die fünf bevölkerungsreichsten Gebiete.

Hilfreiche Ausdrücke: `df.corr()['target']['Population']`, `plt.hist(df['Population'], bins=50)`, `df.nlargest(5, 'Population')` oder `df.sort_values('Population', ascending=False).head(5)`.

### 4. Preiskategorien

Schreibe `price_categories(df)`:
- Teile die Regionen in drei Kategorien ein: `günstig`, `mittel` und `teuer`.
- Zähle die Regionen pro Kategorie.
- Erstelle ein Balkendiagramm der Kategorien.

Hilfreiche Ausdrücke: `pd.qcut(df['target'], q=3, labels=['günstig', 'mittel', 'teuer'])` erzeugt drei ungefähr gleich große Gruppen. `pd.cut(..., bins=3)` erzeugt stattdessen drei gleich breite Preisintervalle. Die Häufigkeiten erhältst du mit `categories.value_counts()` und das Diagramm zum Beispiel mit `plt.bar(categories.index, categories.values)`.

### 5. Merkmale vergleichen

Schreibe `compare_features(df)`:
- Wähle die beiden Merkmale mit der stärksten Korrelation zum Hauswert.
- Erstelle ein 2D-Streudiagramm dieser Merkmale.
- Färbe die Punkte nach Hauswert ein.

Hilfreiche Ausdrücke: `df.corr()['target'].abs().sort_values(ascending=False)`, danach `correlations.drop('target').head(2)`, sowie `plt.scatter(df[feature1], df[feature2], c=df['target'], cmap='viridis')` und `plt.xlabel()`/`plt.ylabel()`.